<a href="https://colab.research.google.com/github/lifeoflifu/Deep-Learning-Labs/blob/main/lab10_agents_and_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DSML 4220 - Lab 10: A simple Agent with Tools

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sgeinitz/DSML4220/blob/main/lab10_agents_and_tools.ipynb)

[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/sgeinitz/DSML4220/blob/main/lab10_agents_and_tools.ipynb)

In this lab we will use Ollama to create a simple agent armed with tools in order to help carry out tasks on our behalf. This notebook is based on the short blog posts/tutorials found [here](https://www.cohorte.co/blog/using-ollama-with-python-step-by-step-guide) and [here](https://towardsdatascience.com/ai-agents-from-zero-to-hero-part-1/).


### Lab 10 Assignment/Task
There are a few questions below that require some additional code to be written so that your agent can carry out other operations besides just addition.

Let's start out by setting up Ollama to run in Colab. If you run this notebook locally and already have Ollama running, then you can skip these steps.

In [ ]:
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,608 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,990 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/multiver

The following two modules we'll need later on, but we install them here because Colab may ask to restart after they are installed with `pip`. It's better to restart at the beginning than to restart half-way through.

In [ ]:
!pip install langchain_community
!pip install -U duckduckgo-search
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 119.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 8.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 4.7 MB/s eta 0:00:00


Now we need to get the Ollama server running. Run the following code block to do this.

In [ ]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

Next, let's pull the model we want to use, Llama 3.2 with 1 billion parameters.

In [ ]:
!ollama pull llama3.2:1b

Then, install the Ollama Python api.

In [ ]:
!pip install ollama

Finally, get started with using Ollama from Python.

In [ ]:
import ollama

Now, let's define a __tool__ for the agent/model to use.

In [ ]:
# Tool function to add two numbers
def add_two_numbers(a: int, b: int) -> int:
    return a + b

Next, let's set up the system prompt and an initial user prompt/question for the agent/model.

In [ ]:
# System prompt to inform the model about the tool is usage
system_message = {
    "role": "system",
    "content": "You are a helpful assistant. You can do math by calling a function 'add_two_numbers' if needed."
}

# A sample of user input asking a math question
user_message = {
    "role": "user",
    "content": "What is 90999999 + 10000001?"
}

messages = [system_message, user_message]
messages

[{'role': 'system',
  'content': "You are a helpful assistant. You can do math by calling a function 'add_two_numbers' if needed."},
 {'role': 'user', 'content': 'What is 90999999 + 10000001?'}]

Ask the agent/model to respond.

In [ ]:
# Ask llama3.2 to respond
response = ollama.chat(
    model='llama3.2:1b',
    messages=messages,
    tools=[add_two_numbers]
)

In [ ]:
response.message

Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='add_two_numbers', arguments={'a': '90999999', 'b': '10000001'}))])

In [ ]:
response.message.content

''

In [ ]:
# Check if the model called a function
if response.message.tool_calls:
    for tool_call in response.message.tool_calls:
        func_name = tool_call.function.name   # e.g., "add_two_numbers"
        args = tool_call.function.arguments   # e.g., {"a": 10, "b": 10}
        # If the function name matches and we have it in our tools, execute it:
        if func_name == "add_two_numbers":
            result = add_two_numbers(**args)
            print("Function output:", result)




Function output: 9099999910000001


---

### Q1: Does the above output look correct? Does it look like the sum of the numbers 90999999 and 10000001? Why is it not correct?

(Hint: there is nothing wrong with the model/agent here, but rather the tool implementation; namely, Python's [type hints](https://docs.python.org/3/library/typing.html) are not a guarantee that the correct/intended data type is used, so you may need to add some type casting inside of the function `add_two_numbers`)

`<No, the output is not correct. The correct sum should be 101,000,000. The earlier output was wrong because the tool received the arguments as strings, such as '90999999' and '10000001'. In Python, using + on strings performs concatenation, so the two values were joined together instead of added. The model/agent did call the tool correctly, but the tool implementation was wrong because Python type hints do not force the input values to become integers. The function should use int(a) + int(b) before returning the result.>`

---

In [ ]:
# Complete the agent's tool call and allow the model to use output to formulate an answer
""" (Continuing from previous code) """
available_functions = {"add_two_numbers": add_two_numbers}#, "multiply_two_numbers": multiply_two_numbers}

""" System prompt to inform the model about the tool is usage """

""" Model's initial response after possibly invoking the tool """
assistant_reply = response.message.content
print("Assistant (initial):", assistant_reply)

""" If a tool was called, handle it """
for tool_call in (response.message.tool_calls or []):
    func = available_functions.get(tool_call.function.name)
    if func:
        result = func(**tool_call.function.arguments)
        # Provide the result back to the model in a follow-up message
        messages.append({"role": "assistant", "content": f"The result is {result}."})
        messages.append({"role": "user", "content": "Can you summarize and state the results you found?"})
        follow_up = ollama.chat(model='llama3.2:1b', messages=messages)
        print("Assistant (final):", follow_up.message.content)

Assistant (initial): 
Assistant (final): I found that I was unable to perform the addition operation between 90999999 and 10000001 in a step-by-step manner as requested, so I will provide a summary of the calculation instead.

The sum of 90999999 and 10000001 is 10101000400000000.


---

### Q2: Try running the code cell below. Does it return the expect result? If note, then add/modify the necessary code to allow Llama3.2 to use its  multiplication tool. Then rerun your code cell below; now did it output the expected result?

`<At first, the code did not return the expected result because the multiplication function was not implemented; it only had pass. I modified multiply_two_numbers so it returns int(a) * int(b). The int() casting is needed because the model may pass the tool arguments as strings even though the function uses int type hints. After rerunning the code, it returned 60006 for 10001 times 6, which is the expected result.E>`

---

In [ ]:
# Implement a multiplication function by replacing the `pass` statement below with the correct return statement
def multiply_two_numbers(a: int, b: int) -> int:
    #pass
    return int(a) * int(b)


""" System prompt to inform the model about the tool is usage """
system_message = {
    "role": "system",
    "content": "You are a helpful assistant. You can do addition by calling the function 'add_two_numbers' or multiplication by calling the function 'multiply_two_numbers'."
}
# User asks a question that involves a calculation
user_message = {
    "role": "user",
    "content": "What is 10001 times 6?"
}

messages = [system_message, user_message]

response = ollama.chat(
    model='llama3.2:1b',
    messages=messages,
    tools=[add_two_numbers, multiply_two_numbers]  # pass the actual function object as a tool
)

# Model's initial reponse after (hopefully) calling the tool
assistant_reply = response.message.content
print("Assistant (initial):", assistant_reply)

# If a tool was called, then handle it
available_functions = {"add_two_numbers": add_two_numbers, "multiply_two_numbers": multiply_two_numbers}
for tool_call in (response.message.tool_calls or []):
    func = available_functions.get(tool_call.function.name)
    if func:
        result = func(**tool_call.function.arguments)
        # Provide the result back to the model in a follow-up message
        messages.append({"role": "assistant", "content": f"The result is {result}."})
        messages.append({"role": "user", "content": "Can you summarize and state the results you found?"})
        follow_up = ollama.chat(model='llama3.2:1b', messages=messages)
        print("Assistant (final):", follow_up.message.content)

Assistant (initial): 
Assistant (final): To solve this problem, I'll use the multiplication function.

I will multiply 10001 by 6:

10001 × 6 = 60006

Here are the results I found:

* The result of multiplying 10001 by 6 is 60006.
* This operation can also be represented as: 10001 * 6 = 60,006


In [ ]:
follow_up.message

Message(role='assistant', content="To solve this problem, I'll use the multiplication function.\n\nI will multiply 10001 by 6:\n\n10001 × 6 = 60006\n\nHere are the results I found:\n\n* The result of multiplying 10001 by 6 is 60006.\n* This operation can also be represented as: 10001 * 6 = 60,006", thinking=None, images=None, tool_name=None, tool_calls=None)

Next let's equip our agent to retrieve external information, which will require a few more tools to be able to search the web.

In [ ]:
from langchain_community.tools import DuckDuckGoSearchResults


def search_web(query: str) -> str:
  return DuckDuckGoSearchResults(backend="news").run(query)

tool_search_web = {'type':'function', 'function':{
  'name': 'search_web',
  'description': 'Search the web',
  'parameters': {'type': 'object',
                'required': ['query'],
                'properties': {
                    'query': {'type':'str', 'description':'the topic or subject to search on the web'},
}}}}

# Quickly test and see what a general web news search for Los Angeles yields
search_web(query="Los Angeles")

"snippet: NBA playoff odds for Los Angeles Lakers vs Houston Rockets Game 6 betting, with point spread, moneyline, over/under for Friday, May 1, 2026., title: Houston Rockets vs Los Angeles Lakers odds for NBA playoffs Game 6, link: https://www.usatoday.com/story/sports/nba/2026/04/29/houston-rockets-los-angeles-lakers-odds-nba-playoff-game-6-betting/89868643007/, date: 2026-04-30T18:04:00+00:00, source: USA TODAY, snippet: Here's how to watch Friday's Los Angeles Angels vs New York Mets game, including start times, TV channels, scores and how to stream., title: Where to watch New York Mets vs Los Angeles Angels: TV channel, start time, streaming for May 1, link: https://www.msn.com/en-us/sports/mlb/where-to-watch-new-york-mets-vs-los-angeles-angels-tv-channel-start-time-streaming-for-may-1/ar-AA22bdRG, date: 2026-05-02T10:38:19+00:00, source: USA TODAY, snippet: Here's how to watch Friday's St. Louis Cardinals vs Los Angeles Dodgers game, including start times, TV channels, scores and

In [ ]:
def search_ys(query: str) -> str:
  engine = DuckDuckGoSearchResults(backend="news")
  return engine.run(f"site:sports.yahoo.com {query}")

tool_search_ys = {'type':'function', 'function':{
  'name': 'search_ys',
  'description': 'Search for sports news',
  'parameters': {'type': 'object',
                'required': ['query'],
                'properties': {
                    'query': {'type':'str', 'description':'the sport, sports team, or subject to search'},
}}}}

# Quickly test and see what a search for Los Angeles in the sports section of the news yields
search_ys(query="Los Angeles")

"snippet: LOS ANGELES– The Miami Marlins defeated the Los Angeles Dodgers, 3-2, at Dodger Stadium on April 29th, 2026 and The Sporting ..., title: TST Images: Marlins defeat Dodgers, 3-2, in Los Angeles, link: https://sports.yahoo.com/articles/tst-images-marlins-defeat-dodgers-230518427.html, date: 2026-04-30T10:56:49+00:00, source: Yahoo Sports, snippet: We’re at the point in the series where the Houston Rockets and Los Angeles Lakers are familiar with one another. There’s not ..., title: Houston Rockets vs. Los Angeles Lakers Game 6 preview, link: https://sports.yahoo.com/articles/houston-rockets-vs-los-angeles-040000313.html, date: 2026-05-01T10:56:49+00:00, source: Yahoo Sports, snippet: The Los Angeles Rams eschewed adding a difference-maker for a Super Bowl run in 2026 in the 2026 NFL Draft, so their draft ..., title: Los Angeles Rams Day 3 2026 NFL Mock Draft: Rams ran out of time to add a 2026 difference-maker, so it's all about depth, link: https://sports.yahoo.com/articles/lo

In [ ]:
system_message = {
    "role": "system",
    "content": "You are a helpful assistant with access to tools for search the web for current news and events."
    }
user_message = {
    "role": "user",
    #"content": "Tell me about the city of Denver." # YOU WILL CHANGE THIS QUESTION, SEE Q3 BELOW
    "content": "Search for current Denver sports news."
}
messages = [system_message, user_message]

In [ ]:
messages

[{'role': 'system',
  'content': 'You are a helpful assistant with access to tools for search the web for current news and events.'},
 {'role': 'user', 'content': 'Search for current Denver sports news.'}]

In [ ]:
response = ollama.chat(
  model="llama3.2:1b",
  tools=[tool_search_web, tool_search_ys],
  messages=messages
)
response

ChatResponse(model='llama3.2:1b', created_at='2026-05-02T11:02:50.969413941Z', done=True, done_reason='stop', total_duration=1095832114, load_duration=1044687039, prompt_eval_count=222, prompt_eval_duration=8204688, eval_count=20, eval_duration=29118612, message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='search_ys', arguments={'query': 'Denver sports news'}))]), logprobs=None)

In [ ]:
# Model's initial reponse after (hopefully) calling the tool
assistant_reply = response.message.content
print("Assistant (initial):", assistant_reply)

# If a tool was called, then handle it
available_functions = {'search_web':search_web, 'search_ys':search_ys}
for tool_call in (response.message.tool_calls or []):
    func = available_functions.get(tool_call.function.name)
    if func:
        result = func(**tool_call.function.arguments)
        # Provide the result back to the model in a follow-up message
        messages.append({"role": "assistant", "content": f"The result is {result}."})
        messages.append({"role": "user", "content": "Can you summarize and state the results you found?"})
        follow_up = ollama.chat(model='llama3.2:1b', messages=messages)
        print("Assistant (final):", follow_up.message.content)

Assistant (initial): 
Assistant (final): Based on the search results, here is a summary of the current Denver sports news:

* The Minnesota Timberwolves are leading the series against the Denver Nuggets 3-1.
* Keith Abney II is considered a sleeper pick for the Detroit Lions in the NFL draft.
* Anthony Edwards is not playing for the Denver Nuggets due to an injury.
* The Denver Nuggets will look to bounce back from losing Game 2 of their playoff series against the Minnesota Timberwolves.

Unfortunately, I didn't find any recent news on the Denver Broncos or Colorado Avalanche. However, I did find some general news articles about the teams and the playoffs.

Here are the specific results I found:

* The article on Keith Abney II being a sleeper pick for the Detroit Lions was from NFL.com's Gennaro Filice and was published on April 28, 2026.
* The article on Anthony Edwards not playing for the Denver Nuggets due to injury was from Yahoo Sports and was published on April 28, 2026.
* The a

---

### Q3: The question above currently asks about Denver, but change the question to include a word or reference to sports. Does the agent use the correct tool based on your prompt/question? Be sure to also run the code cells above with your modified promp/question.

`<I changed the question from “Tell me about the city of Denver” to “Search for current Denver sports news.” After rerunning the code cells, the agent selected the search_ys tool with the query “Denver sports news.” This was the correct tool because search_ys is designed to search for sports news, while search_web is the more general web search tool. The final response summarized Denver sports-related search results, so yes, the agent used the correct tool based on my modified prompt.>`

---